# VinDr-CXR DINO Swin-L Training in Google Colab

This notebook wraps the local training pipeline in this repository for Colab. It assumes your repo folder already contains:

- `vincxr/` with the JPG images and annotation CSVs
- `checkpoints/checkpoint0027_5scale_swin.pth`
- the training scripts and config from this repo

The easiest setup is to put the whole repo folder in Google Drive and point `REPO_DIR` at it below.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/swindino_vincxr'  # @param {type:"string"}
REPO_DIR = Path(REPO_DIR)
assert REPO_DIR.exists(), f'Missing repo directory: {REPO_DIR}'

%cd {REPO_DIR}
print('Using repo root:', REPO_DIR.resolve())
!ls

In [ ]:
# Install the MMDetection stack into the current Colab runtime.
# Colab already ships with torch; openmim will resolve compatible mmcv wheels.
!pip install -q -U openmim pycocotools pillow
!mim install -q "mmengine>=0.10.0,<1.0.0"
!mim install -q "mmcv>=2.0.0,<2.2.0"
!pip install -q "mmdet>=3.2.0,<3.4.0"

In [ ]:
required_paths = [
    REPO_DIR / 'configs' / 'vindr_dino_swinl_36e.py',
    REPO_DIR / 'scripts' / 'prepare_vindr_cxr.py',
    REPO_DIR / 'scripts' / 'train_vindr.py',
    REPO_DIR / 'scripts' / 'eval_vindr.py',
    REPO_DIR / 'vincxr',
    REPO_DIR / 'checkpoints' / 'checkpoint0027_5scale_swin.pth',
]
for path in required_paths:
    assert path.exists(), f'Missing required path: {path}'

print('Repo layout looks valid.')

In [ ]:
%cd {REPO_DIR}
!python scripts/prepare_vindr_cxr.py

import json
stats = json.loads((REPO_DIR / 'artifacts' / 'vindr_cxr' / 'stats.json').read_text())
print(json.dumps(stats['data'], indent=2))
print(json.dumps(stats['labels']['cluster_support_hist'], indent=2))

In [ ]:
%cd {REPO_DIR}
!python scripts/visualize_vindr_sample.py --split val --sample 6

from IPython.display import Image as ColabImage, display

viz_dir = REPO_DIR / 'artifacts' / 'visualizations' / 'val'
for image_path in sorted(viz_dir.glob('*_annotated.png'))[:6]:
    display(ColabImage(filename=str(image_path)))

In [ ]:
%cd {REPO_DIR}
CONFIG = 'configs/vindr_dino_swinl_36e.py'
WORK_DIR = 'work_dirs/colab_vindr_dino_swinl'  # @param {type:"string"}

# Single-GPU Colab training.
!python scripts/train_vindr.py $CONFIG --work-dir $WORK_DIR

In [ ]:
%cd {REPO_DIR}
CONFIG = 'configs/vindr_dino_swinl_36e.py'
WORK_DIR = 'work_dirs/colab_vindr_dino_swinl'  # @param {type:"string"}
RESUME_FROM = ''  # @param {type:"string"}

if RESUME_FROM.strip():
    !python scripts/train_vindr.py $CONFIG --work-dir $WORK_DIR --resume $RESUME_FROM
else:
    !python scripts/train_vindr.py $CONFIG --work-dir $WORK_DIR --resume

In [ ]:
%cd {REPO_DIR}
from pathlib import Path

CONFIG = 'configs/vindr_dino_swinl_36e.py'
WORK_DIR = 'work_dirs/colab_vindr_dino_swinl'  # @param {type:"string"}
SPLIT = 'test'  # @param ['val', 'test']

WORK_DIR = Path(WORK_DIR)
best_candidates = sorted(WORK_DIR.glob('best_coco_bbox_mAP*.pth'))
if best_candidates:
    checkpoint_path = best_candidates[-1]
elif (WORK_DIR / 'last_checkpoint').exists():
    checkpoint_path = Path((WORK_DIR / 'last_checkpoint').read_text().strip())
else:
    raise FileNotFoundError('No checkpoint found in the selected work dir.')

print('Evaluating checkpoint:', checkpoint_path)
!python scripts/eval_vindr.py $CONFIG $checkpoint_path --split $SPLIT